In [ ]:
!apt-get update -y

# 1) Install Google Chrome from official .deb
!wget -q -O /tmp/google-chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y /tmp/google-chrome.deb

# 2) Install Selenium + webdriver-manager
!pip install -U selenium webdriver-manager

In [ ]:
!apt-get update -y
!apt-get install -y tor
!pip install stem requests[socks]

In [ ]:
!which google-chrome
!google-chrome --version

In [ ]:
# ==============================
#
# WORKSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS
#
# FULLY INTEGRATED TOR + SELENIUM VOTING BOT FOR GOOGLE COLAB
# Changes IP after EVERY successful vote + shows current IP
# ==============================

import os
import time
import random
import socket
import subprocess
import requests
from IPython.display import clear_output

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

from stem import Signal
from stem.control import Controller

# =====================
# CONFIGURATION
# =====================
TOR_CONTROL_PASSWORD = "my_very_strong_password_2025"  # Change this!

POLL_URL = "https://entermedia.io/city/itogi-goda-2025-gorod-lyudi-i-komandy/#kommunikatsiya_goda"
POLL_CONTAINER_ID = "PDI_container16335706"
TEAM_TEXT = "МУП «Водоканал» Казани"
SUBMIT_BUTTON_ID = "pd-vote-button16335706"

# =====================
# 1. INSTALL TOR & SETUP TOR
# =====================
print("Installing Tor and dependencies...")
!apt-get update -y > /dev/null 2>&1
!apt-get install -y tor > /dev/null 2>&1
!pip install stem requests[socks] webdriver-manager > /dev/null 2>&1

clear_output(wait=False)

# =====================
# 2. TOR CONFIGURATION
# =====================
def generate_torrc():
    hashed = subprocess.check_output(
        ["tor", "--hash-password", TOR_CONTROL_PASSWORD]
    ).decode().strip().splitlines()[-1]

    torrc_content = f"""
SocksPort 9050
ControlPort 9051
HashedControlPassword {hashed}
DataDirectory ./tor_data
Log notice stdout
AvoidDiskWrites 1
"""
    with open("torrc", "w") as f:
        f.write(torrc_content)

def start_tor_process():
    generate_torrc()
    print("Starting Tor daemon...")
    proc = subprocess.Popen(
        ["tor", "-f", "torrc"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    # Wait until Tor bootstraps
    for line in proc.stdout:
        print(line.strip())
        if "Bootstrapped 100%" in line:
            print("Tor is fully bootstrapped!")
            break
    return proc

# =====================
# 3. TOR IP TOOLS
# =====================
def get_current_ip():
    proxies = {
        "http": "socks5h://127.0.0.1:9050",
        "https": "socks5h://127.0.0.1:9050",
    }
    try:
        r = requests.get("https://api.ipify.org", proxies=proxies, timeout=12)
        return r.text.strip()
    except:
        return "Unknown"

def rotate_ip():
    print("Requesting new Tor circuit (NEWNYM)...", end=" ")
    try:
        with Controller.from_port(port=9051) as controller:
            controller.authenticate(password=TOR_CONTROL_PASSWORD)
            controller.signal(Signal.NEWNYM)
        print("Done")
        time.sleep(8)  # Wait for new circuit
    except Exception as e:
        print(f"Failed: {e}")

# =====================
# 4. SELENIUM SETUP WITH TOR PROXY
# =====================
def create_tor_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1280,2400")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-infobars")
    options.add_argument("--start-maximized")

    # Random real-looking User-Agent
    uas = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/129.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/129.0 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/129.0 Safari/537.36",
    ]
    options.add_argument(f"--user-agent={random.choice(uas)}")

    # Tor proxy settings
    options.add_argument("--proxy-server=socks5://127.0.0.1:9050")
    # Bypass Cloudflare/Tor detection
    options.add_argument("--disable-extensions")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    # Hide webdriver property
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => false});")
    return driver

# =====================
# 5. VOTING FUNCTION
# =====================
def safe_click(driver, element):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
    time.sleep(0.4)
    driver.execute_script("arguments[0].click();", element)

def cast_single_vote(vote_num):
    driver = None
    try:
        current_ip = get_current_ip()
        print(f"\nVote #{vote_num + 1} | Current Tor IP: {current_ip}")

        driver = create_tor_driver()
        wait = WebDriverWait(driver, 20)

        driver.get(POLL_URL)
        print("Page loaded")

        # Wait for poll container
        wait.until(EC.presence_of_element_located((By.ID, POLL_CONTAINER_ID)))
        time.sleep(2)  # Let Polldaddy JS render

        # Find and click the desired option
        xpath = f"//div[@id='{POLL_CONTAINER_ID}']//label[descendant::span[contains(text(), '{TEAM_TEXT}')]]"
        option_label = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
        safe_click(driver, option_label)
        print(f"Selected: {TEAM_TEXT}")

        # Click vote button
        vote_btn = wait.until(EC.element_to_be_clickable((By.ID, SUBMIT_BUTTON_ID)))
        safe_click(driver, vote_btn)
        print("Vote submitted!")

        time.sleep(3)

        new_ip = get_current_ip()
        print(f"Vote #{vote_num + 1} SUCCESS with IP: {current_ip} -> {new_ip if new_ip != current_ip else '(same)'}")
        return True

    except Exception as e:
        print(f"Vote #{vote_num + 1} FAILED: {e}")
        return False
    finally:
        if driver:
            driver.quit()

# =====================
# 6. MAIN EXECUTION
# =====================
# Start Tor once
if not os.path.exists("torrc"):
    tor_process = start_tor_process()
else:
    print("Tor config exists – assuming Tor is running.")

# Initial IP
time.sleep(5)
print(f"\nInitial Tor Exit IP: {get_current_ip()}\n")
time.sleep(3)

# Ask how many votes
while True:
    try:
        total_votes = int(input("How many votes do you want to cast? "))
        if total_votes > 0:
            break
        print("Enter a positive number.")
    except:
        print("Invalid input.")

print(f"\nStarting {total_votes} votes – changing IP after each successful one...\n")
time.sleep(2)

successful = 0
for i in range(total_votes):
    if cast_single_vote(i):
        successful += 1
        if i < total_votes - 1:  # Don't rotate after last vote
            rotate_ip()

    # Anti-detection delay
    delay = random.uniform(4, 9)
    print(f"Waiting {delay:.1f}s before next vote...\n")
    time.sleep(delay)

print(f"\nFinished! {successful}/{total_votes} votes successful.")
print(f"Final Tor IP: {get_current_ip()}")